# RAG 고도화 1단계 재현

- 기준: 1단계 최종 채택 상태를 재현합니다.
- 기본선: `B-02`
- 조건부 보강: `B-03a`
- Ceiling check: `B-06 (gpt-5)`
- 제외: RFP 파싱 원본 생성, 미채택 실험(B-04, B-05.x)의 상세 재현


In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()  # 필요 시 리포지토리 루트 절대경로로 수정
PROCESSED_DOCS_PATH = PROJECT_ROOT / "processed_data" / "processed_documents.jsonl"
QUESTION_SET_PATH = PROJECT_ROOT / "evaluation" / "day3_partA_eval_questions_v1.txt"
CHROMA_DIR = PROJECT_ROOT / "rag_outputs" / "chroma_db"
RAG_OUTPUTS = PROJECT_ROOT / "rag_outputs"

PYTHON = sys.executable

def run_cmd(args, cwd=PROJECT_ROOT):
    print("[RUN]", " ".join(str(x) for x in args))
    result = subprocess.run(
        [str(x) for x in args],
        cwd=str(cwd),
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"command failed: {result.returncode}")
    return result

def show_csv(path, n=20):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    display(df.head(n))
    return df

print("PROJECT_ROOT =", PROJECT_ROOT)
print("CHROMA_DIR   =", CHROMA_DIR)



In [ ]:
required_paths = [PROCESSED_DOCS_PATH, QUESTION_SET_PATH, PROJECT_ROOT / '.env']
for path in required_paths:
    print(path, '->', path.exists())


## 1. B-02 Prefix-v2 Chunk 생성

In [ ]:
B02_CHUNK_PATH = RAG_OUTPUTS / 'b02_prefix_v2_chunks.jsonl'
B02_CHUNK_SUMMARY = RAG_OUTPUTS / 'b02_prefix_v2_chunk_summary.csv'
B02_DOC_FIELDS = RAG_OUTPUTS / 'b02_prefix_v2_document_fields.csv'
B02_MANIFEST = RAG_OUTPUTS / 'b02_prefix_v2_manifest.json'
run_cmd([
    PYTHON, PROJECT_ROOT / '10_B02_prefix_v2_청킹.py',
    '--입력경로', PROCESSED_DOCS_PATH,
    '--출력경로', B02_CHUNK_PATH,
    '--요약CSV경로', B02_CHUNK_SUMMARY,
    '--문서필드CSV경로', B02_DOC_FIELDS,
    '--manifest경로', B02_MANIFEST,
    '--청크크기', '500',
    '--겹침크기', '80',
])
show_csv(B02_CHUNK_SUMMARY)


## 2. B-02 Embedding + Chroma 적재

In [ ]:
B02_COLLECTION = 'rfp_contextual_chunks_v2_b02'
run_cmd([
    PYTHON, PROJECT_ROOT / '02_임베딩_생성_크로마적재.py',
    '--청크경로', B02_CHUNK_PATH,
    '--크로마경로', CHROMA_DIR,
    '--컬렉션이름', B02_COLLECTION,
    '--기존컬렉션초기화',
])
show_csv(RAG_OUTPUTS / 'chroma_적재_요약.csv')


## 3. B-02 BM25 인덱스 생성

In [ ]:
B02_BM25_INDEX = RAG_OUTPUTS / 'bm25_index_b02_notebook.pkl'
B02_BM25_MANIFEST = RAG_OUTPUTS / 'bm25_index_b02_notebook_manifest.json'
run_cmd([
    PYTHON, PROJECT_ROOT / '07_B01_BM25_인덱스생성.py',
    '--청크경로', B02_CHUNK_PATH,
    '--출력경로', B02_BM25_INDEX,
    '--manifest경로', B02_BM25_MANIFEST,
])
print(B02_BM25_INDEX)


## 4. B-02 Full 평가(병렬)

In [ ]:
B02_EVAL_DIR = RAG_OUTPUTS / 'b02_full_eval_parallel_notebook'
run_cmd([
    PYTHON, PROJECT_ROOT / '28_평가_샤딩_실행.py',
    '--question-set-path', QUESTION_SET_PATH,
    '--collection-name', B02_COLLECTION,
    '--bm25-index-path', B02_BM25_INDEX,
    '--chroma-dir', CHROMA_DIR,
    '--response-model', 'gpt-5-mini',
    '--judge-model', 'gpt-5',
    '--output-dir', B02_EVAL_DIR,
    '--shard-count', '4',
    '--workers', '4',
])
show_csv(B02_EVAL_DIR / 'baseline_eval_manual_summary.csv')


## 5. 현재 채택 파이프라인(B-02 default + B-03 conditional) 평가

In [ ]:
B06_MINI_DIR = RAG_OUTPUTS / 'b06_adopted_eval_gpt5mini_notebook'
run_cmd([
    PYTHON, PROJECT_ROOT / '34_B06_채택파이프라인_평가.py',
    '--question-set-path', QUESTION_SET_PATH,
    '--response-model', 'gpt-5-mini',
    '--judge-model', 'gpt-5',
    '--output-dir', B06_MINI_DIR,
    '--shard-count', '4',
    '--workers', '4',
])
show_csv(B06_MINI_DIR / 'baseline_eval_manual_summary.csv')


## 6. B-06 Ceiling Check (`gpt-5`)

In [ ]:
B06_GPT5_DIR = RAG_OUTPUTS / 'b06_adopted_eval_gpt5_notebook'
run_cmd([
    PYTHON, PROJECT_ROOT / '34_B06_채택파이프라인_평가.py',
    '--question-set-path', QUESTION_SET_PATH,
    '--response-model', 'gpt-5',
    '--judge-model', 'gpt-5',
    '--output-dir', B06_GPT5_DIR,
    '--shard-count', '4',
    '--workers', '4',
])
show_csv(B06_GPT5_DIR / 'baseline_eval_manual_summary.csv')


## 7. B-06 비교표 생성

In [ ]:
B06_COMPARE_CSV = RAG_OUTPUTS / 'b06_compare_notebook.csv'
B06_COMPARE_JSON = RAG_OUTPUTS / 'b06_compare_notebook.json'
run_cmd([
    PYTHON, PROJECT_ROOT / '35_B06_비교.py',
    '--mini-dir', B06_MINI_DIR,
    '--gpt5-dir', B06_GPT5_DIR,
    '--out-csv', B06_COMPARE_CSV,
    '--out-json', B06_COMPARE_JSON,
])
show_csv(B06_COMPARE_CSV)


## 메모

- 현재 1단계 채택 기준은 `B-02 default + B-03 conditional`입니다.
- `B-04`, `B-05`, `B-05.1`은 전역 채택하지 않았습니다.
- `B-05.2`는 표 질문 subset에서만 유망했고, 2단계 후보로 넘깁니다.
